In [10]:
%pip install torch
%pip install transformers
%pip install datasets
%pip install peft
%pip install bitsandbytes
%pip install accelerate
%pip install evaluate

Python(72594) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Python(72605) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Python(72614) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Python(72645) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Python(72655) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#imports
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, concatenate_datasets

In [ ]:
#Global Variables
LORA = False
QUANTIZE = False
CLUSTER = False
CUDA = True
CPU = False
DEBUG = False
MODEL = "Qwen/Qwen3-4B"
# MODEL = "Qwen/Qwen3-8B"

In [13]:
device = torch.device("cuda" if CUDA and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [ ]:
tasks = [
    "narrativeqa", "qasper", "multifieldqa_en", "multifieldqa_zh", 
    "hotpotqa", "2wikimqa", "musique", "dureader", "gov_report", 
    "qmsum", "multi_news", "vcsum", "trec", "triviaqa", "samsum",
    "lsht", "passage_count", "passage_retrieval_en", "passage_retrieval_zh",
    "lcc", "repobench-p"
]
datasets = {}
for task in tasks:
    if CLUSTER:
        datasets[task] = load_dataset("THUDM/LongBench", task, split="test", cache_dir='/home/rhayrapetyan/.cache/huggingface/datasets/downloads/extracted/0fe2e4841dad9538f66462ef845f9ebb8d619462c2028d55ba8593b7c60267f6/data/downloads/extracted/0fe2e4841dad9538f66462ef845f9ebb8d619462c2028d55ba8593b7c60267f6/data/')
    else:
        datasets[task] = load_dataset("THUDM/LongBench", task, split="test")

Using the latest cached version of the dataset since THUDM/LongBench couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'narrativeqa' at /Users/rubenhayrapetyan/.cache/huggingface/datasets/THUDM___long_bench/narrativeqa/1.0.0/4a916a4bde5c3481ac49b84d5dde69a9d2eefcd67f884ef65b3d97ee7cc91f3e (last modified on Mon Oct 20 18:56:39 2025).
Using the latest cached version of the dataset since THUDM/LongBench couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'qasper' at /Users/rubenhayrapetyan/.cache/huggingface/datasets/THUDM___long_bench/qasper/1.0.0/4a916a4bde5c3481ac49b84d5dde69a9d2eefcd67f884ef65b3d97ee7cc91f3e (last modified on Sat Oct 18 18:21:57 2025).
Using the latest cached version of the dataset since THUDM/LongBench couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'multifieldqa_en' at /Users/rubenhayrapetyan/.cache/huggingface/datasets/THUDM___long_bench/multifieldqa_en

In [ ]:
#dataset generation
def format(training_example):
    prompt = (
        "Given the following context, answer the question concisely.\n\n"
        f"Question: {training_example['input']}\n"
        f"Context: {training_example['context']}\n\n"
    )
    answer = training_example["answers"]

    if isinstance(answer, list):
        answer = answer[0]
    elif answer == [""] or "":
        raise ValueError("Empty answer found in the dataset.")

    return {"prompt": prompt, "response": answer}

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(training_example):
    prompt_tokens = tokenizer(training_example["prompt"], padding="max_length", truncation=True)['input_ids']
    if DEBUG:
        print(type(prompt_tokens))
        print(prompt_tokens)
    answer_tokens = tokenizer(training_example["response"], padding="max_length", truncation=True)['input_ids']
    if DEBUG:
        print(type(answer_tokens))
        print(answer_tokens)
    input_ids = prompt_tokens + answer_tokens + [tokenizer.eos_token_id]
    if DEBUG:
        print(type(input_ids))
        print(input_ids)
    labels = [-100] * len(prompt_tokens) + answer_tokens + [tokenizer.eos_token_id]
    if DEBUG:
        print(type(labels))
        print(labels)
    attention_mask = [1] * len(input_ids)
    if DEBUG:
        print(type(attention_mask))
        print(attention_mask)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

In [16]:
train_datasets = {}
for task_name, ds in datasets.items():
    train_datasets[task_name] = ds.map(format, remove_columns=ds.column_names)
all_train_dataset = concatenate_datasets(list(train_datasets.values()))
tokenized_dataset = all_train_dataset.map(tokenize, remove_columns=all_train_dataset.column_names)

In [17]:
train, test = tokenized_dataset.train_test_split(test_size=0.1).values()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

model = None
print("Loading model...")

model = None
bnb_config = None

if QUANTIZE:
    print("  Applying 4-bit quantization config...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config if QUANTIZE else None,
    device_map="auto" if CUDA else None,
    trust_remote_code=True
)

if LORA:
    print("  Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_config)
else:
    model = base_model

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("Model ready ✅\n")
print(model)

for name, module in model.named_modules():
    print(name)


Loading model...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from evaluate import load

metric = load("rouge")

def compute_metrics(predictions, references):
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_refs, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return result

In [ ]:
from transformers import TrainingArguments

mode = []
if QUANTIZE and LORA:
    mode.append("quantized_lora")
if QUANTIZE:
    mode.append("quantized")
if LORA:
    mode.append("lora")
if not mode:
    mode.append("fp")  # full precision, no quantization, no LoRA

output_dir = f"./Qwen3-8B-LongBench-finetuned-{'_'.join(mode)}"

training_args = TrainingArguments(
    output_dir=output_dir,
    do_eval=True,
    eval_strategy="steps",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    save_steps=600,
    eval_steps=300,
    logging_steps=300,
    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=False,
    use_cpu=CPU
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=lambda data: {
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['input_ids']) for f in data], batch_first=True, padding_value=tokenizer.pad_token_id
        ),
        'attention_mask': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['attention_mask']) for f in data], batch_first=True, padding_value=0
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['labels']) for f in data], batch_first=True, padding_value=-100
        ),
    },
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer
)



/var/folders/lv/pdfd1q0x6qd7nsw0pkqndf700000gn/T/ipykernel_66805/1563030330.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


RuntimeError: MPS backend out of memory (MPS allocated: 71.44 GB, other allocations: 592.00 KB, max allowed: 81.60 GB). Tried to allocate 32.34 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).